In [1]:
from pathlib import Path
import sys

project_dir = Path.cwd()

if not all(
    (project_dir / filename).is_file()
    for filename in ["README.md", "data/evaluation/development_v1.json"]
):
    raise RuntimeError("Open this notebook from the project root.")

print("Project folder:", project_dir)
print("Python executable:", sys.executable)
print("Failure-handling lab ready.")

Project folder: C:\Users\Khaleel\workplace-ai-operations-lab
Python executable: C:\Users\Khaleel\workplace-ai-operations-lab\.venv\Scripts\python.exe
Failure-handling lab ready.


In [2]:
import re

STOP_WORDS = {
    "a", "an", "the", "is", "are", "do", "does", "how",
    "what", "which", "of", "to", "for", "per", "in",
    "on", "and", "or", "with",
}


class ConfigurationError(ValueError):
    """The retrieval configuration is invalid."""


class SourceDataError(RuntimeError):
    """The configured document source is unavailable or malformed."""


def validate_top_k(top_k):
    # Reject bool explicitly: Python treats bool as a subclass of int.
    if type(top_k) is not int or not 1 <= top_k <= 10:
        raise ConfigurationError(
            "top_k must be an integer between 1 and 10."
        )


def load_policy_passages(source_dir):
    source_dir = Path(source_dir)

    if not source_dir.is_dir():
        raise SourceDataError("Document directory is unavailable.")

    try:
        files = sorted(source_dir.glob("*.txt"))

        if not files:
            raise SourceDataError("No policy text files were found.")

        loaded = []

        for file_path in files:
            text = file_path.read_text(encoding="utf-8")
            _, separator, body = text.partition("\n\n")

            if not separator or not body.strip():
                raise SourceDataError(
                    f"Missing policy body: {file_path.name}"
                )

            for line_number, line in enumerate(body.splitlines(), start=1):
                sentence = line.strip()

                if sentence:
                    loaded.append({
                        "source": file_path.name,
                        "passage_id": f"{file_path.stem}:{line_number}",
                        "text": sentence,
                        "tokens": (
                            set(re.findall(r"[a-z0-9]+", sentence.lower()))
                            - STOP_WORDS
                        ),
                    })

    except (OSError, UnicodeError) as error:
        raise SourceDataError("Policy files could not be read.") from error

    return loaded


validate_top_k(2)

loaded_passages = load_policy_passages(
    project_dir / "data" / "synthetic"
)

print("Configuration accepted: top_k=2")
print("Policy passages loaded:", len(loaded_passages))

Configuration accepted: top_k=2
Policy passages loaded: 9


In [3]:
import json
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4


class RequestValidationError(ValueError):
    """The query is invalid."""


# Separate log file for each execution of this setup cell.
log_dir = project_dir / "evidence" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

log_path = log_dir / f"retrieval_requests_{uuid4().hex}.jsonl"


def retrieve_request(question, source_dir, top_k=2):
    request_id = uuid4().hex
    started = perf_counter()

    results = []
    status = "error"
    error_code = None

    try:
        validate_top_k(top_k)

        if not isinstance(question, str) or not question.strip():
            raise RequestValidationError("A non-empty query is required.")

        query_tokens = (
            set(re.findall(r"[a-z0-9]+", question.lower()))
            - STOP_WORDS
        )

        if not query_tokens:
            raise RequestValidationError("Query has no searchable tokens.")

        request_passages = load_policy_passages(source_dir)

        for passage in request_passages:
            matched_words = query_tokens & passage["tokens"]

            if matched_words:
                results.append({
                    "passage_id": passage["passage_id"],
                    "text": passage["text"],
                    "score": len(matched_words),
                })

        results.sort(
            key=lambda item: (-item["score"], item["passage_id"])
        )
        results = results[:top_k]
        status = "ok" if results else "no_match"

    except ConfigurationError:
        error_code = "INVALID_CONFIGURATION"
    except RequestValidationError:
        error_code = "INVALID_QUERY"
    except SourceDataError:
        error_code = "SOURCE_UNAVAILABLE_OR_INVALID"

    elapsed_ms = round((perf_counter() - started) * 1000, 3)

    event = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "request_id": request_id,
        "event": "retrieval_completed",
        "status": status,
        "error_code": error_code,
        "result_count": len(results),
        "elapsed_ms": elapsed_ms,
    }

    # Log metadata only, excluding query text and retrieved content.
    # A log-write failure propagates rather than silently losing evidence.
    with log_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(event) + "\n")

    return {
        **event,
        "results": results,
    }


response = retrieve_request(
    "How many annual leave days do full-time employees receive?",
    project_dir / "data" / "synthetic",
)

print("Status:", response["status"])
print("Request ID:", response["request_id"])
print("Results returned:", response["result_count"])
print("Top passage:", response["results"][0]["passage_id"])
print("Log file:", log_path)

Status: ok
Request ID: 626ff33e75694c4898a3dc472bd9b706
Results returned: 2
Top passage: leave_policy:1
Log file: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\logs\retrieval_requests_2d1ea04ed92c41368e127e0d7f3301b2.jsonl


In [4]:
source_dir = project_dir / "data" / "synthetic"
missing_dir = project_dir / f"missing_source_{uuid4().hex}"

if missing_dir.exists():
    raise RuntimeError("The simulated missing directory unexpectedly exists.")

checks = [
    {
        "name": "No matching passages",
        "question": "What is the cafeteria breakfast menu?",
        "source": source_dir,
        "top_k": 2,
        "expected_status": "no_match",
        "expected_error": None,
    },
    {
        "name": "Missing source directory",
        "question": "How many annual leave days are available?",
        "source": missing_dir,
        "top_k": 2,
        "expected_status": "error",
        "expected_error": "SOURCE_UNAVAILABLE_OR_INVALID",
    },
    {
        "name": "Invalid top_k",
        "question": "How many annual leave days are available?",
        "source": source_dir,
        "top_k": 0,
        "expected_status": "error",
        "expected_error": "INVALID_CONFIGURATION",
    },
    {
        "name": "Boolean top_k",
        "question": "How many annual leave days are available?",
        "source": source_dir,
        "top_k": True,
        "expected_status": "error",
        "expected_error": "INVALID_CONFIGURATION",
    },
    {
        "name": "Empty query",
        "question": "   ",
        "source": source_dir,
        "top_k": 2,
        "expected_status": "error",
        "expected_error": "INVALID_QUERY",
    },
]

checked_responses = []

for check in checks:
    result = retrieve_request(
        check["question"],
        check["source"],
        top_k=check["top_k"],
    )

    assert result["status"] == check["expected_status"], check["name"]
    assert result["error_code"] == check["expected_error"], check["name"]
    assert result["result_count"] == 0, check["name"]
    assert result["results"] == [], check["name"]

    checked_responses.append(result)

    print(
        f"PASS: {check['name']} | "
        f"status={result['status']} | "
        f"error={result['error_code']}"
    )

# Confirm every tested request has exactly one matching log event.
events = [
    json.loads(line)
    for line in log_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

for result in checked_responses:
    matching_events = [
        event for event in events
        if event["request_id"] == result["request_id"]
    ]

    assert len(matching_events) == 1
    assert matching_events[0] == {
        key: value for key, value in result.items()
        if key != "results"
    }

print("\nChecks passed:", len(checked_responses))
print("Each tested request has one matching log event.")

PASS: No matching passages | status=no_match | error=None
PASS: Missing source directory | status=error | error=SOURCE_UNAVAILABLE_OR_INVALID
PASS: Invalid top_k | status=error | error=INVALID_CONFIGURATION
PASS: Boolean top_k | status=error | error=INVALID_CONFIGURATION
PASS: Empty query | status=error | error=INVALID_QUERY

Checks passed: 5
Each tested request has one matching log event.


In [5]:
from collections import Counter

events = [
    json.loads(line)
    for line in log_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# Verify the successful request from the earlier cell.
successful_events = [
    event for event in events
    if event["request_id"] == response["request_id"]
]

assert len(successful_events) == 1
assert successful_events[0] == {
    key: value for key, value in response.items()
    if key != "results"
}

# Validate the metadata-only log schema.
expected_fields = {
    "timestamp_utc", "request_id", "event", "status",
    "error_code", "result_count", "elapsed_ms",
}

for event in events:
    assert set(event) == expected_fields
    assert event["elapsed_ms"] >= 0

request_ids = [event["request_id"] for event in events]
assert len(request_ids) == len(set(request_ids))

status_counts = Counter(event["status"] for event in events)

print("Logged requests:", len(events))

for status in ["ok", "no_match", "error"]:
    print(f"{status}: {status_counts[status]}")

print("Successful-request log verified.")
print("All events have the expected metadata fields.")
print("All request IDs are unique.")

Logged requests: 6
ok: 1
no_match: 1
error: 4
Successful-request log verified.
All events have the expected metadata fields.
All request IDs are unique.
